# Assembling a beam line

Real hardware is a sequence of devices: a beam pipe, a bellows, a cavity,
another bellows, an absorber. `cavsim2d` lets you concatenate them with `+`.
The result is an `Assembly`, and an `Assembly` is itself a cavity — it meshes,
runs eigenmode and wakefield analyses, tunes and optimises exactly like a
single geometry.

We will

1. build a line and look at it,
2. see how its parameters are named,
3. check that concatenating costs nothing,
4. set the boundary condition at the two ends,
5. tune one element while it sits in the line.

This notebook is live: every figure below is produced by running `cavsim2d`.

In [ ]:
import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jn_zeros

from cavsim2d import Assembly, BLA, Beampipe, Bellows, EllipticalCavity
from cavsim2d.utils.style import apply_style, WARM

apply_style()
workspace = tempfile.mkdtemp()

# A TESLA-like mid cell: A, B, a, b, Ri, L, Req  (mm)
MIDCELL = [62.22, 66.13, 30.22, 23.11, 80.0, 93.5, 171.20]
RI = MIDCELL[4]

## 1. Building a line

Add the devices together in beam order. Each element keeps whatever beam pipe
its own parameterisation carries, so the cavity here is built with
`beampipe='none'` and the drift is supplied by explicit `Beampipe` elements —
what you write is what you get.

In [ ]:
def bellows():
    return Bellows(Ri=RI, A=15.0, L_p=10.0, N_conv=6, R_root=2.0, R_crest=2.0)

line = (Beampipe(R=RI, L=187.0, name='bp_in')
        + bellows()
        + EllipticalCavity(1, MIDCELL, beampipe='none', name='cav')
        + bellows()
        + BLA(R=RI, L=150.0, absorber_length=80.0, absorber_thickness=8.0,
              eps_r=10.0, tan_delta=0.3, maxh=4.0))
line.set_name('line')

print(repr(line))
print('elements :', line.labels)
print(f'length   : {line.length:.1f} mm')

In [ ]:
ax = line.plot('geometry', mirror=True, show=False)
ax.legend(loc='center', fontsize=8, ncols=2)
plt.show()

The two bellows were both called `bellows`, so they were given distinct labels
`bellows_1` and `bellows_2`. Any element is reachable by its label.

In [ ]:
print(line.element('bellows_2').parameters)
print('cavity cells :', line.element('cav').n_cells)

## 2. Parameters are namespaced by element

An assembly gathers its elements' parameters into one flat dictionary, keyed
`'<label>:<name>'`. That string is the handle everywhere else: a tune config, an
optimisation variable, a UQ variable.

In [ ]:
print(f'{len(line.parameters)} parameters in total, for example:\n')
for key in ('bp_in:L', 'bellows_1:A', 'bellows_1:R_crest',
            'cav:Req_m', 'bla:absorber_thickness'):
    print(f'  {key:26s} = {line.parameters[key]}')

In [ ]:
# A bare name still means what it means to the element that owns it: on an
# elliptical cavity 'Req' is the same quantity in every cell.
print(line.expand_variable('cav:Req'))

# Reading and writing go straight through to the element.
line.set_tune_value('bellows_1:A', 12.0)
print('bellows_1 depth now :', line.element('bellows_1').parameters['A'], 'mm')
line.set_tune_value('bellows_1:A', 15.0)

Each element still owns its own constraints. Push a corner radius past what
fits and the bellows says so, naming the quantity — an optimiser searching this
space gets a meaningful rejection rather than a meshing failure.

In [ ]:
line.parameters['bellows_1:R_crest'] = 3.0
try:
    line.profile()
except ValueError as err:
    print(err)
line.parameters['bellows_1:R_crest'] = 2.0

## 3. Concatenating costs nothing

Joining two devices removes the aperture at each side of the junction: the
walls meet and the vacuum becomes one region. There is no interface plane, and
nothing is inserted.

The check is direct — build a cavity with its pipes the ordinary way, build the
same thing by concatenation, and compare.

In [ ]:
L_bp = 2 * MIDCELL[5]

monolithic = EllipticalCavity(1, MIDCELL, beampipe='both', name='monolithic')
assembled = Assembly([Beampipe(R=RI, L=L_bp, name='l'),
                      EllipticalCavity(1, MIDCELL, beampipe='none', name='c'),
                      Beampipe(R=RI, L=L_bp, name='r')], name='assembled')

a = np.asarray(monolithic.profile().contour_points(2e-4, skip=('AXI',)))
b = np.asarray(assembled.profile().contour_points(2e-4, skip=('AXI',)))
a[:, 0] -= a[:, 0].min()
b[:, 0] -= b[:, 0].min()
print('same number of points :', a.shape == b.shape)
print('largest difference    :', np.abs(a - b).max(), 'm')

In [ ]:
cfg = {'polarisation': 'monopole', 'n_modes': 3,
       'boundary_conditions': 'mm', 'mesh_config': {'h': 8, 'p': 2}}

for cav in (monolithic, assembled):
    cav.set_workspace(os.path.join(workspace, cav.name))
    cav.eigenmode.run(cfg)
    q = cav.eigenmode.qois
    print(f"{cav.name:11s} f = {q['freq [MHz]']:9.4f} MHz   "
          f"R/Q = {q['R/Q [Ohm]']:6.2f} Ohm   ff = {q['ff [%]']:6.2f} %")

Identical, as it must be. So any difference you see when you *do* add a device
is that device's physics, not an artefact of assembling.

In [ ]:
line.set_workspace(os.path.join(workspace, 'line'))
line.eigenmode.run(cfg)
q = line.eigenmode.qois
b = monolithic.eigenmode.qois
print(f"bare cavity          f = {b['freq [MHz]']:9.4f} MHz   "
      f"R/Q = {b['R/Q [Ohm]']:6.2f} Ohm   ff = {b['ff [%]']:6.2f} %")
print(f"with bellows + BLA   f = {q['freq [MHz]']:9.4f} MHz   "
      f"R/Q = {q['R/Q [Ohm]']:6.2f} Ohm   ff = {q['ff [%]']:6.2f} %")
print(f"\nshift: {q['freq [MHz]'] - b['freq [MHz]']:+.3f} MHz, "
      f"{q['R/Q [Ohm]'] - b['R/Q [Ohm]']:+.2f} Ohm")

## 4. The two ends

Only the outer ends of a line are boundaries at all, and `set_ends` decides what
they are. It overrides whatever the end elements declared, in either direction —
it can open an end an element built closed.

A two-pipe line makes this checkable: shorted at both ends it is a closed
cylinder, whose TM$_{010}$ frequency is known exactly.

In [ ]:
R = 80.0
f_analytic = 299792458.0 * jn_zeros(0, 1)[0] / (2 * np.pi * R * 1e-3) * 1e-6

for ends in ('pec', 'pmc'):
    # note the first element was built closed; the assembly's ends win
    tube = Assembly([Beampipe(R=R, L=100.0, ends='pec', name='a'),
                     Beampipe(R=R, L=100.0, name='b')],
                    ends=ends, name=f'tube_{ends}')
    tube.set_workspace(os.path.join(workspace, tube.name))
    tube.eigenmode.run({'polarisation': 'monopole', 'n_modes': 3,
                        'boundary_conditions': 'mm',
                        'mesh_config': {'h': 6, 'p': 3}})
    print(f"ends = {ends} :  f = {tube.eigenmode.qois['freq [MHz]']:9.3f} MHz")

print(f"\nanalytic closed cylinder TM010 = {f_analytic:.3f} MHz")

## 5. Tuning an element inside the line

Tuning takes the namespaced variable name. Everything else is the ordinary tune
config, and the tuned line comes back as an `Assembly`.

One practical note: ask for a frequency tolerance your mesh can actually deliver.
A beam line is a much larger mesh than a single cavity, and re-meshing it each
iteration moves the answer by ~1 kHz, so a `tol` tighter than that just spends
iterations chasing noise. 10 kHz (`tol=1e-2` MHz) is far finer than any tuning
you would trust from a 2D model.

In [ ]:
target = 800.0

tuned_line = (Beampipe(R=RI, L=187.0, name='bp_in')
              + bellows()
              + EllipticalCavity(1, MIDCELL, beampipe='none', name='cav')
              + Beampipe(R=RI, L=187.0, name='bp_out'))
tuned_line.set_name('tuned_line')
tuned_line.set_workspace(os.path.join(workspace, 'tuned_line'))

before = tuned_line.parameters['cav:Req_m']
tuned_line.tune.run({'freqs': target,
                     'cell_type': {'mid-cell': 'cav:Req'},
                     'tol': 1e-2,                      # MHz
                     'eigenmode_config': cfg})

res = tuned_line.tune.qois['mid-cell']
print(f"\ntarget      : {target:.3f} MHz")
print(f"achieved    : {res['FREQ']:.3f} MHz")
print(f"cav:Req     : {before:.3f} -> {tuned_line.parameters['cav:Req_m']:.3f} mm")

## 6. The absorber

`BLA` is a beam pipe with a lossy ring set into its wall.

| parameter | meaning |
| --- | --- |
| `R` | bore radius — the free aperture, unchanged by the ring |
| `L` | overall element length |
| `absorber_length` | axial length of the ring |
| `absorber_thickness` | radial thickness, measured outward from the bore: the ring spans `R` to `R + absorber_thickness` |
| `z_absorber` | axial centre of the ring, from the element centre (default 0) |
| `eps_r`, `tan_delta` | permittivity and loss tangent of the absorber material |
| `maxh` | mesh size inside the ring |

![BLA parameters](../../_static/beamline_bla_nomenclature.png)

In an assembly the ring is reported in line coordinates and its material name is
prefixed with the element label, so two absorbers in one line stay distinct.

In [ ]:
bore = line.element('bla').parameters['R']
for d in line.dielectrics:
    print(f"{d['material']:16s} z = {d['z'][0]:7.1f} .. {d['z'][1]:7.1f} mm"
          f"   r = {d['r'][0]:5.1f} .. {d['r'][1]:5.1f} mm"
          f"   eps_r = {d['eps_r']}, tan d = {d['tan_delta']}")
print()
print(f"bore radius {bore:.1f} mm; the ring starts at r = {d['r'][0]:.1f} mm, "
      "so the aperture is untouched.")

## What to take away

* `a + b + c` builds an `Assembly`, and an `Assembly` behaves like any cavity.
* Parameters are `'<label>:<name>'`; that is the handle for tuning, optimisation
  and UQ, and each element keeps enforcing its own constraints.
* Junctions are not boundaries — an assembled cavity reproduces the monolithic
  one exactly.
* `set_ends` controls the two outer faces, and overrides the end elements.
* Use `chain=` instead when you want *one* cavity repeated into a module; that
  is a different operation and it is unchanged.

Next: [The taper element](taper.ipynb) for matching two bores, and
[A multi-cavity module](multi_cavity_module.ipynb) for a nine-element line with
three cavities and its fundamental passband.